# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
print(2)

2


In [2]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [3]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [4]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:40<00:00,  8.00s/it]


In [5]:
len(deals)

50

In [6]:
deals[44].describe()

'Title: Yitahome 5-Tier S-Shaped Bookshelf for $64 + free shipping\nDetails: Apply coupon code "DNBC04" for an extra savings of $16. It\'s available in several colors (Grey pictured). Buy Now at Yitahome\nFeatures: \nURL: https://www.dealnews.com/Yitahome-5-Tier-S-Shaped-Bookshelf-for-64-free-shipping/21757615.html?iref=rss-c196'

In [7]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [8]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [9]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Jackery Explorer 1000 Plus 1,264Wh Portable Power Station + SolarSaga 100W Mini Solar Panel for $650 + free shipping
Details: That's a savings of $150. Buy Now at Costco
Features: 2000W output and 4000W peak 1,264Wh capacity, expandable to 5kWh Charges fully in 100 minutes Multiple device charging ports
URL: https://www.dealnews.com/Jackery-Explorer-1000-Plus-1-264-

In [10]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [11]:
result = get_recommendations()

In [12]:
len(result.deals)

5

In [13]:
result.deals[1]

Deal(product_description='The Jackery Explorer 5000 Plus stands out as a highly efficient portable power station, boasting a capacity of 5,040Wh with a continuous output of 7,200W. This device is designed for heavy-duty usage, supporting dual voltage of 120V/240V and the ability to accommodate up to 5 expandable battery packs. It features multiple outlets to cater to various devices simultaneously, making it ideal for extended outdoor trips or home backup power. This power station ensures reliability with its substantial energy output and durability.', price=4599.0, url='https://www.dealnews.com/Jackery-Explorer-5000-Plus-5-040-Wh-7-200-W-Portable-Power-Station-w-Solar-Saga-500-X-Solar-Panel-2-Pack-for-4-599-free-shipping/21757763.html?iref=rss-c196')

In [14]:
from agents.scanner_agent import ScannerAgent

In [15]:
agent = ScannerAgent()
result = agent.scan()

In [16]:
result

DealSelection(deals=[Deal(product_description='The Jackery Explorer 1000 Plus is a versatile portable power station with a capacity of 1,264Wh, designed to power a variety of devices with its 2000W output and 4000W peak power. It allows multiple devices to charge simultaneously through various ports and can be fully charged in just 100 minutes. Additionally, it can be expanded to hold up to 5kWh, making it suitable for longer trips or outdoor activities. This model also comes with a SolarSaga 100W Mini Solar Panel, providing an eco-friendly charging option.', price=650.0, url='https://www.dealnews.com/Jackery-Explorer-1000-Plus-1-264-Wh-Portable-Power-Station-Solar-Saga-100-W-Mini-Solar-Panel-for-650-free-shipping/21757690.html?iref=rss-c196'), Deal(product_description='The Jackery Explorer 5000 Plus is a high-capacity portable power station featuring 5,040Wh and an impressive 7,200W continuous output with a peak output of 14,400W. This unit supports dual voltage of 120V/240V and can a